In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install wandb
!pip install wandb

import torch
import torch.nn as nn
import numpy as np
import wandb
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

paths = {
    "NURBS": (
        "/content/drive/MyDrive/LDC Dataset/nurbs_lid_driven_cavity_X.npz",
        "/content/drive/MyDrive/LDC Dataset/nurbs_lid_driven_cavity_Y.npz"
    ),
    "Harmonics": (
        "/content/drive/MyDrive/LDC Dataset/harmonics_lid_driven_cavity_X.npz",
        "/content/drive/MyDrive/LDC Dataset/harmonics_lid_driven_cavity_Y.npz"
    ),
    "Skelneton": (
        "/content/drive/MyDrive/LDC Dataset/skelneton_lid_driven_cavity_X.npz",
        "/content/drive/MyDrive/LDC Dataset/skelneton_lid_driven_cavity_Y.npz"
    )
}

In [ ]:
def create_coords(H, W, device):
    """
    Creates normalized (x, y) coordinates in [0, 1]^2.
    Output shape: [H*W, 2]
    """
    x = torch.linspace(0, 1, H)
    y = torch.linspace(0, 1, W)
    grid_x, grid_y = torch.meshgrid(x, y, indexing='ij')
    coords = torch.stack([grid_x, grid_y], dim=-1).reshape(-1, 2)  # [H*W, 2]
    return coords.to(device)

In [ ]:
class MLP(nn.Module):
    """
    Standard fully-connected MLP with ReLU activations.
    Final layer is linear (no activation).
    """
    def __init__(self, in_dim, hidden_dim, out_dim, num_layers=4):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden_dim), nn.ReLU()]
        for _ in range(num_layers - 2):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU()]
        layers.append(nn.Linear(hidden_dim, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class DeepONet(nn.Module):
    """
    Vanilla DeepONet.

    Branch: encodes flattened input field  -> latent vector [B, latent_dim]
    Trunk:  encodes (x, y) coords         -> [N, 3 * latent_dim]
            (one latent block per output channel: u, v, p)

    Output via einsum: output[b, n, c] = sum_k branch[b, k] * trunk[n, c, k] + bias[c]
    """
    def __init__(self, branch_dim, coord_dim=2, latent_dim=256, hidden_dim=256, num_layers=4):
        super().__init__()
        self.latent_dim = latent_dim

        self.branch = MLP(branch_dim, hidden_dim, latent_dim, num_layers=num_layers)
        self.trunk  = MLP(coord_dim, hidden_dim, latent_dim * 3, num_layers=num_layers)  # 3 output channels

        # Learnable per-channel bias
        self.bias = nn.Parameter(torch.zeros(3))

    def forward(self, branch_input, coords):
        branch_out = self.branch(branch_input)                    # [B, latent_dim]
        trunk_out  = self.trunk(coords)                           # [N, 3*latent_dim]
        trunk_out  = trunk_out.view(-1, 3, self.latent_dim)       # [N, 3, latent_dim]

        output = torch.einsum("bk,nck->bnc", branch_out, trunk_out)  # [B, N, 3]
        output = output + self.bias
        return output

In [ ]:
class Normalizer:
    """
    Z-score normalizer fitted on training data.
    """
    def __init__(self, data, eps=1e-8):
        self.mean = data.mean(dim=0, keepdim=True)
        self.std  = data.std(dim=0, keepdim=True).clamp(min=eps)

    def encode(self, x):
        return (x - self.mean.to(x.device)) / self.std.to(x.device)

    def decode(self, x):
        return x * self.std.to(x.device) + self.mean.to(x.device)

In [ ]:
wandb.login()

In [ ]:
def train_geometry(geometry_name, x_path, y_path):
    print(f"\n==== Training on {geometry_name} ====")

    # ── Load & validate data ──────────────────────────────────────────────────
    X_data = np.load(x_path)["data"]
    Y_data = np.load(y_path)["data"]
    assert Y_data.shape[1] >= 3, "Y must have at least 3 channels (u, v, p)"
    Y_data = Y_data[:, 0:3, :, :]

    X = torch.tensor(X_data, dtype=torch.float32)
    Y = torch.tensor(Y_data, dtype=torch.float32)

    # ── Train / Val split ─────────────────────────────────────────────────────
    X_train, X_val, Y_train, Y_val = train_test_split(
        X, Y, test_size=0.2, random_state=42
    )

    # ── Fit normalizers on TRAINING data only ─────────────────────────────────
    N, C, H, W = X_train.shape

    X_flat_train = X_train.reshape(N, -1)
    X_flat_val   = X_val.reshape(X_val.shape[0], -1)

    x_norm = Normalizer(X_flat_train)
    X_flat_train = x_norm.encode(X_flat_train)
    X_flat_val   = x_norm.encode(X_flat_val)

    Y_flat_train = Y_train.reshape(Y_train.shape[0], 3, -1)  # [N, 3, H*W]
    y_norm = Normalizer(Y_flat_train)

    # ── DataLoaders ───────────────────────────────────────────────────────────
    train_loader = DataLoader(
        TensorDataset(X_flat_train, Y_train),
        batch_size=16, shuffle=True, pin_memory=True
    )
    val_loader = DataLoader(
        TensorDataset(X_flat_val, Y_val),
        batch_size=16, shuffle=False, pin_memory=True
    )

    # ── Coords (plain x,y, fixed) ─────────────────────────────────────────────
    coords = create_coords(H, W, device)  # [H*W, 2]

    # ── Model, optimizer, scheduler ───────────────────────────────────────────
    branch_dim = C * H * W
    latent_dim = 256
    hidden_dim = 256
    num_layers = 4
    epochs     = 100
    lr         = 3e-4

    model     = DeepONet(
        branch_dim=branch_dim,
        coord_dim=2,
        latent_dim=latent_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )
    mse = nn.MSELoss()

    # ── W&B init ──────────────────────────────────────────────────────────────
    wandb.init(
        project="DeepONet-LDC-Vanilla",
        group="DeepONet-Vanilla-v1",
        name=f"{geometry_name}-run",
        config={
            "geometry":      geometry_name,
            "epochs":        epochs,
            "lr":            lr,
            "latent_dim":    latent_dim,
            "hidden_dim":    hidden_dim,
            "num_layers":    num_layers,
            "resolution":    H,
            "activation":    "ReLU",
            "normalization": "z-score",
            "coord_dim":     2,
        }
    )

    best_val_l2 = float("inf")
    save_path   = f"/content/drive/MyDrive/LDC Dataset/{geometry_name}_vanilla_best.pt"

    # ── Training loop ─────────────────────────────────────────────────────────
    for epoch in range(epochs):
        model.train()
        train_num, train_den = 0.0, 0.0
        total_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)   # [B, branch_dim], normalized
            yb = yb.to(device, non_blocking=True)   # [B, 3, H, W], raw scale

            # Normalize targets
            yb_flat = yb.reshape(yb.shape[0], 3, -1)
            yb_norm = y_norm.encode(yb_flat)             # [B, 3, H*W]
            yb_norm_spatial = yb_norm.permute(0, 2, 1)  # [B, H*W, 3]

            pred = model(xb, coords)                     # [B, H*W, 3]
            loss = mse(pred, yb_norm_spatial)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            train_num  += torch.norm(pred - yb_norm_spatial).item()
            train_den  += torch.norm(yb_norm_spatial).item()

        scheduler.step()
        train_loss = total_loss / len(train_loader)
        train_l2   = train_num / (train_den + 1e-8)

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        val_num, val_den = 0.0, 0.0
        val_loss_sum = 0.0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                yb_flat = yb.reshape(yb.shape[0], 3, -1)
                yb_norm = y_norm.encode(yb_flat)
                yb_norm_spatial = yb_norm.permute(0, 2, 1)

                pred = model(xb, coords)
                val_loss_sum += mse(pred, yb_norm_spatial).item()
                val_num      += torch.norm(pred - yb_norm_spatial).item()
                val_den      += torch.norm(yb_norm_spatial).item()

        val_loss = val_loss_sum / len(val_loader)
        val_l2   = val_num / (val_den + 1e-8)

        # ── Checkpoint best model ─────────────────────────────────────────────
        if val_l2 < best_val_l2:
            best_val_l2 = val_l2
            torch.save({
                "epoch":       epoch,
                "model":       model.state_dict(),
                "optimizer":   optimizer.state_dict(),
                "val_l2":      best_val_l2,
                "x_norm_mean": x_norm.mean,
                "x_norm_std":  x_norm.std,
                "y_norm_mean": y_norm.mean,
                "y_norm_std":  y_norm.std,
            }, save_path)

        wandb.log({
            "epoch":        epoch,
            "train_loss":   train_loss,
            "val_loss":     val_loss,
            "train_rel_L2": train_l2,
            "val_rel_L2":   val_l2,
            "lr":           scheduler.get_last_lr()[0],
            "best_val_L2":  best_val_l2,
        })

        if epoch % 20 == 0:
            print(f"{geometry_name} | Epoch {epoch:3d} | "
                  f"Train L2: {train_l2:.4f} | Val L2: {val_l2:.4f} | "
                  f"Best: {best_val_l2:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    print(f"\n{geometry_name} training done. Best Val L2: {best_val_l2:.4f}")
    wandb.finish()


for geometry_name, (x_path, y_path) in paths.items():
    train_geometry(geometry_name, x_path, y_path)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import torch


def visualize_predictions(geometry_name, model, x_norm, y_norm, X_val, Y_val, coords, H, W, device, num_samples=3):
    """
    Generates streamline plots comparing ground truth vs predicted flow fields.
    Visualizes U velocity, V velocity, Pressure, and Streamlines side by side.
    """
    model.eval()
    indices = np.random.choice(len(X_val), size=num_samples, replace=False)

    for idx in indices:
        xb = X_val[idx:idx+1].to(device)   # [1, branch_dim], already normalized
        yb = Y_val[idx:idx+1].to(device)   # [1, 3, H, W], raw scale

        with torch.no_grad():
            pred_norm = model(xb, coords)   # [1, H*W, 3]

        # ── Denormalize prediction ────────────────────────────────────────────
        pred_norm_t = pred_norm.permute(0, 2, 1)    # [1, 3, H*W]
        pred_raw    = y_norm.decode(pred_norm_t)    # [1, 3, H*W]
        pred_field  = pred_raw.reshape(1, 3, H, W)  # [1, 3, H, W]

        pred_np = pred_field[0].cpu().numpy()   # [3, H, W]
        true_np = yb[0].cpu().numpy()           # [3, H, W]

        U_pred, V_pred, P_pred = pred_np[0], pred_np[1], pred_np[2]
        U_true, V_true, P_true = true_np[0], true_np[1], true_np[2]

        speed_pred = np.sqrt(U_pred**2 + V_pred**2)
        speed_true = np.sqrt(U_true**2 + V_true**2)

        x_lin = np.linspace(0, 1, W)
        y_lin = np.linspace(0, 1, H)
        X_grid, Y_grid = np.meshgrid(x_lin, y_lin)  # [H, W]

        # ── Figure: 4 rows x 3 cols ───────────────────────────────────────────
        fig, axes = plt.subplots(4, 3, figsize=(18, 22))
        fig.suptitle(
            f"{geometry_name} — Sample {idx}\nTop→Bottom: U velocity | V velocity | Pressure | Streamlines",
            fontsize=14, fontweight='bold', y=0.98
        )

        fields = [
            ("U Velocity", U_true, U_pred, "RdBu_r"),
            ("V Velocity", V_true, V_pred, "RdBu_r"),
            ("Pressure",   P_true, P_pred, "viridis"),
        ]

        # ── Rows 0–2: scalar fields ───────────────────────────────────────────
        for row, (label, true_f, pred_f, cmap) in enumerate(fields):
            error_f = np.abs(pred_f - true_f)
            vmin = min(true_f.min(), pred_f.min())
            vmax = max(true_f.max(), pred_f.max())

            im0 = axes[row, 0].imshow(true_f, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
            axes[row, 0].set_title(f"{label} — Ground Truth", fontsize=11)
            plt.colorbar(im0, ax=axes[row, 0], fraction=0.046, pad=0.04)

            im1 = axes[row, 1].imshow(pred_f, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
            axes[row, 1].set_title(f"{label} — Predicted", fontsize=11)
            plt.colorbar(im1, ax=axes[row, 1], fraction=0.046, pad=0.04)

            im2 = axes[row, 2].imshow(error_f, origin='lower', cmap='hot_r', aspect='equal')
            axes[row, 2].set_title(f"{label} — |Error|  (max={error_f.max():.3e})", fontsize=11)
            plt.colorbar(im2, ax=axes[row, 2], fraction=0.046, pad=0.04)

            for ax in axes[row]:
                ax.set_xticks([]); ax.set_yticks([])

        # ── Row 3: Streamlines ────────────────────────────────────────────────
        streamline_configs = [
            ("Ground Truth Streamlines", U_true, V_true, speed_true),
            ("Predicted Streamlines",    U_pred, V_pred, speed_pred),
            ("Speed Error |‖u‖ - ‖û‖|", None,   None,   np.abs(speed_true - speed_pred)),
        ]

        for col, (title, U, V, speed) in enumerate(streamline_configs):
            ax = axes[3, col]
            if col < 2:
                strm = ax.streamplot(
                    X_grid, Y_grid, U, V,
                    color=speed, cmap='plasma', linewidth=1.2,
                    density=1.8, arrowsize=1.2,
                    norm=mcolors.Normalize(vmin=speed.min(), vmax=speed.max())
                )
                plt.colorbar(strm.lines, ax=ax, fraction=0.046, pad=0.04, label='Speed')
                ax.axhline(y=1.0, color='red', linewidth=2.0, linestyle='--', label='Lid')
                ax.legend(fontsize=8, loc='upper right')
            else:
                im = ax.imshow(speed, origin='lower', cmap='hot_r', aspect='equal')
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Speed Error')

            ax.set_title(title, fontsize=11)
            ax.set_xlabel("x"); ax.set_ylabel("y")

        plt.tight_layout(rect=[0, 0, 1, 0.97])

        save_fig_path = f"/content/drive/MyDrive/LDC Dataset/{geometry_name}_sample{idx}_streamlines.png"
        plt.savefig(save_fig_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_fig_path}")

        wandb.log({f"{geometry_name}/streamline_sample_{idx}": wandb.Image(fig)})
        plt.show()
        plt.close(fig)


def run_visualization(geometry_name, x_path, y_path, checkpoint_path, device, num_samples=3):
    """
    Loads the best saved checkpoint and runs streamline visualization
    on random validation samples.
    """
    print(f"\n==== Visualizing {geometry_name} ====")

    X_data = np.load(x_path)["data"]
    Y_data = np.load(y_path)["data"][:, 0:3, :, :]
    X = torch.tensor(X_data, dtype=torch.float32)
    Y = torch.tensor(Y_data, dtype=torch.float32)

    _, X_val, _, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

    N_full, C, H, W = X.shape

    ckpt = torch.load(checkpoint_path, map_location=device)

    class RestoredNormalizer:
        def __init__(self, mean, std):
            self.mean = mean
            self.std  = std
        def encode(self, x):
            return (x - self.mean.to(x.device)) / self.std.to(x.device)
        def decode(self, x):
            return x * self.std.to(x.device) + self.mean.to(x.device)

    x_norm = RestoredNormalizer(ckpt["x_norm_mean"], ckpt["x_norm_std"])
    y_norm = RestoredNormalizer(ckpt["y_norm_mean"], ckpt["y_norm_std"])

    branch_dim = C * H * W
    model = DeepONet(
        branch_dim=branch_dim,
        coord_dim=2,
        latent_dim=256,
        hidden_dim=256,
        num_layers=4
    ).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()

    coords = create_coords(H, W, device)

    X_val_flat = X_val.reshape(X_val.shape[0], -1)
    X_val_norm = x_norm.encode(X_val_flat)

    wandb.init(
        project="DeepONet-LDC-Vanilla",
        group="DeepONet-Vanilla-v1-VIZ",
        name=f"{geometry_name}-viz"
    )

    visualize_predictions(
        geometry_name=geometry_name,
        model=model,
        x_norm=x_norm,
        y_norm=y_norm,
        X_val=X_val_norm,
        Y_val=Y_val,
        coords=coords,
        H=H, W=W,
        device=device,
        num_samples=num_samples
    )

    wandb.finish()


# ── Run for all geometries ────────────────────────────────────────────────────
for geometry_name, (x_path, y_path) in paths.items():
    checkpoint_path = f"/content/drive/MyDrive/LDC Dataset/{geometry_name}_vanilla_best.pt"
    run_visualization(geometry_name, x_path, y_path, checkpoint_path, device, num_samples=3)